# Tratamento da base de dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import json
import os
import numpy as np
import datetime

In [ ]:
base = pd.read_csv('petr4_treinamento_ex.csv')
base

In [ ]:
from datetime import datetime
dateparse = lambda dates: datetime.strptime(dates, '%Y-%m-%d')
base = pd.read_csv('petr4_treinamento_ex.csv', parse_dates = ['Date'], index_col = 'Date', date_parser = dateparse)
base

In [ ]:
base.drop(columns = ['Open', 'High', 'Low', 'Adj Close', 'Volume'], axis = 1, inplace = True)
base

In [ ]:
base_teste = base.copy()
base_teste.shape

In [ ]:
base_treinamento = base.iloc[0:1315,:]
base_treinamento

In [ ]:
base_treinamento.index.min(), base_treinamento.index.max()

In [ ]:
base_teste.index.min(), base_teste.index.max()

In [ ]:
dt_data_inicial = pd.Timestamp("2013-01-02", freq = 'D')
dt_data_final_treinamento = pd.Timestamp("2018-12-04", freq = 'D')
dt_previsao_maxima = pd.Timestamp("2018-05-25", freq = 'D')

faixa_treinamento = (dt_data_inicial, dt_data_final_treinamento)
faixa_teste = (dt_data_inicial, dt_previsao_maxima)

In [ ]:
faixa_treinamento

In [ ]:
faixa_teste

In [ ]:
serie_temporal_treinamento = []
serie_temporal_teste = []

serie_temporal_treinamento.append(base_treinamento[faixa_treinamento[0]:faixa_treinamento[1]]['Close'])
serie_temporal_teste.append(base_teste[faixa_teste[0]:faixa_teste[1]]['Close'])

In [ ]:
type(serie_temporal_treinamento), type(serie_temporal_teste)

In [ ]:
serie_temporal_treinamento

In [ ]:
serie_temporal_teste

In [ ]:
len(serie_temporal_treinamento[0]), len(serie_temporal_teste[0])

In [ ]:
plt.figure(figsize=(10,10))
serie_temporal_teste[0].plot(label = "Teste")
serie_temporal_treinamento[0].plot(label = "Treinamento")
plt.legend()
plt.show()

In [ ]:
def encode_target(ts):
    return [x if np.isfinite(x) else "NaN" for x in ts]

In [ ]:
encode_target(serie_temporal_treinamento[0][0:5])

In [ ]:
def series_to_obj(ts):
    obj = {"start": str(ts.index[0]), "target": encode_target(ts)}
    return obj

In [ ]:
series_to_obj(serie_temporal_treinamento[0][0:5])

In [ ]:
def series_to_jsonline(ts):
    return json.dumps(series_to_obj(ts))

In [ ]:
series_to_jsonline(serie_temporal_treinamento[0][0:5])

In [ ]:
encoding = 'utf-8'
arquivo_json_treinamento = 'train_petr.json'
arquivo_json_teste = 'test_petr.json'
with open(arquivo_json_treinamento, 'wb') as fp:
    for ts in serie_temporal_treinamento:
        fp.write(series_to_jsonline(ts).encode(encoding))
        fp.write('\n'.encode(encoding))

with open(arquivo_json_teste, 'wb') as fp:
    for ts in serie_temporal_teste:
        fp.write(series_to_jsonline(ts).encode(encoding))
        fp.write('\n'.encode(encoding))

# Configura??es do SageMaker

In [ ]:
import boto3
import sagemaker

In [ ]:
session = sagemaker.Session()
role = sagemaker.get_execution_role()
job_name = 'deepar-petr'
bucket = 'cursoawssagemaker'
subpasta_dataset = 'datasets/petr'
subpasta_modelo = 'modelos/petr/deepar'
s3_train_data = 's3://{}/{}/train/{}'.format(bucket, subpasta_dataset, 'train_petr.json')
s3_test_data = 's3://{}/{}/test/{}'.format(bucket, subpasta_dataset, 'test_petr.json')
output_location = 's3://{}/{}/output'.format(bucket, subpasta_modelo)

print('Role: {}'.format(role))
print('Localiza??o da base de treinamento: {}'.format(s3_train_data))
print('Localiza??o da base de teste: {}'.format(s3_test_data))
print('Modelo ser? salvo em: {}'.format(output_location))

In [ ]:
def escreve_s3(arquivo, bucket, nome):
    with open(arquivo, 'rb') as f:
        return boto3.Session().resource('s3').Bucket(bucket).Object(nome).upload_fileobj(f)

In [ ]:
escreve_s3('train_petr.json', bucket, os.path.join(subpasta_dataset, 'train', 'train_petr.json'))
escreve_s3('test_petr.json', bucket, os.path.join(subpasta_dataset, 'test', 'test_petr.json'))

# Treinamento do DeepAR

In [ ]:
# https://docs.aws.amazon.com/sagemaker/latest/dg/ecr-sa-east-1.html
container = sagemaker.image_uris.retrieve(framework = 'forecasting-deepar', region = session.boto_region_name)

In [ ]:
# https://sagemaker.readthedocs.io/en/stable/api/training/estimators.html
# https://github.com/aws-samples/amazon-sagemaker-managed-spot-training/blob/main/xgboost_built_in_managed_spot_training_checkpointing/xgboost_built_in_managed_spot_training_checkpointing.ipynb
deepar = sagemaker.estimator.Estimator(image_uri = container,
                                       role = role,
                                       instance_count = 1,
                                       instance_type = 'ml.m5.large',
                                       output_path = output_location,
                                       sagemaker_session = session,
                                       base_job_name = job_name,
                                       use_spot_instances = True,
                                       max_run = 3600,
                                       max_wait = 3600)

In [ ]:
# https://docs.aws.amazon.com/sagemaker/latest/dg/deepar_hyperparameters.html
freq = 'D'
prediction_length = 30
context_length = 30

In [ ]:
hyperparametros = {
    'time_freq': freq,
    'epochs': "20",
    'context_length': str(context_length),
    'prediction_length': str(prediction_length)
}

In [ ]:
deepar.set_hyperparameters(**hyperparametros)

In [ ]:
data_channels = {'train': s3_train_data, 'test': s3_test_data}
data_channels

In [ ]:
deepar.fit(inputs = data_channels)

# Deploy e previs?es

In [ ]:
job_name = deepar.latest_training_job.name
print('Job name: ', job_name)

In [ ]:
endpoint = session.endpoint_from_job(job_name = job_name,
                                     initial_instance_count = 1,
                                     instance_type = "ml.m5.large",
                                     image_uri = container,
                                     role = role)

In [ ]:
print('Endpoint: ', endpoint)

In [ ]:
class DeepARPredictor(sagemaker.predictor.Predictor):

    def set_prediction_parameters(self, freq, prediction_length):
        """Set the time frequency and prediction length parameters. 
          This method **must** be called before being able to use `predict`.
        
        Parameters:
        freq -- string indicating the time frequency
        prediction_length -- integer, number of predicted time points
        
        Return value: none.
        """
        self.freq = freq
        self.prediction_length = prediction_length
        
    def predict(self, ts, cat=None, dynamic_feat=None, 
                encoding="utf-8", num_samples=100, quantiles=["0.1", "0.5", "0.9"]):
        """Requests the prediction of for the time series listed in `ts`, 
           each with the (optional) corresponding category listed in `cat`.
        
        Parameters:
        ts -- list of `pandas.Series` objects, the time series to predict
        cat -- list of integers (default: None)
        encoding -- string, encoding to use for the request (default: "utf-8")
        num_samples -- integer, number of samples to compute at prediction time (default: 100)
        quantiles -- list of strings specifying the quantiles to compute (default: ["0.1", "0.5", "0.9"])
        
        Return value: list of `pandas.DataFrame` objects, each containing the predictions
        """
        prediction_times = [x.index[-1] + datetime.timedelta(hours=1) for x in ts] 
        
        req = self.__encode_request(ts, cat, dynamic_feat, encoding, num_samples, quantiles)
        res = super(DeepARPredictor, self).predict(req)
        return self.__decode_response(res, prediction_times, encoding)
    
    def __encode_request(self, ts, cat, dynamic_feat, encoding, num_samples, quantiles):
        
        instances = [series_to_obj(ts[k]) for k in range(len(ts))]
        
        configuration = {"num_samples": num_samples, "output_types": ["quantiles"], "quantiles": quantiles}
        http_request_data = {"instances": instances, "configuration": configuration}
        return json.dumps(http_request_data).encode(encoding)
    
    def __decode_response(self, response, prediction_times, encoding):
        response_data = json.loads(response.decode(encoding))
        list_of_df = []
        for k in range(len(prediction_times)):
            prediction_index = pd.date_range(start=prediction_times[k], freq=self.freq, periods=self.prediction_length)
            list_of_df.append(pd.DataFrame(data=response_data['predictions'][k]['quantiles'], index=prediction_index))
        return list_of_df

In [ ]:
predictor = DeepARPredictor(endpoint_name = 'deepar-petr-2022-05-23-17-00-32-832',
                            sagemaker_session = session)
predictor.set_prediction_parameters(freq, prediction_length)
predictor.serializer.content_type = "application/json"

In [ ]:
import datetime
previsoes = predictor.predict(serie_temporal_treinamento)

In [ ]:
len(previsoes[0])

In [ ]:
previsoes

In [ ]:
for k in range(len(previsoes)):
    plt.figure(figsize=(12,6))
    previsoes[k]['0.5'].plot(label = 'Mediana', color = 'red')
    #serie_temporal_teste[k].plot(label = 'Alvo')
    serie_temporal_teste[k][-prediction_length - context_length:].plot(label = 'Alvo')
    p10 = previsoes[k]['0.1']
    p90 = previsoes[k]['0.9']
    plt.fill_between(p10.index, p10, p90, color = 'y', alpha = 0.5, label = 'Intervalo de confian?a')
    plt.legend()
    plt.show()

In [ ]:
df_teste = base.iloc[1315:,:]
len(df_teste)

In [ ]:
df_teste

In [ ]:
previsoes2 = previsoes[0]['0.5'].values
previsoes2

In [ ]:
esperado = df_teste['Close'].values
esperado

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
mae = mean_absolute_error(esperado, previsoes2)
mse = mean_squared_error(esperado, previsoes2)
print('MAE: ', mae, '\nMSE: ', mse)